<div align="center">

<a href="https://colab.research.google.com/github/utkukose/ai-applications-in-engineering-NB-lecture/blob/main/weeks/week-13/NB13_transformers_llms_genai.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>

# Week 13: Attention, Transformers and Generative AI

**Artificial Intelligence Applications in Engineering (MUH-920 Mühendislikte Yapay Zeka Uygulamaları)**<br>Prof. Dr. Utku Kose<br>Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye<br>Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University<br>Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)<br>[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)<br>[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

<sub>Content licensed under CC BY 4.0, code under MIT. This course is updated in line with current developments in the field. Last update: September 2026.</sub>

</div>

## How to use this notebook

This notebook is a self-contained version of the week: The lecture text comes first, followed by the hands-on lab. Run the cells in order with Shift+Enter. No earlier Python experience is assumed: Each new piece of Python appears in a cell marked **Python move**, which explains the syntax right where it is first needed. Exercise cells start with `None` placeholders and print feedback through `check(...)`. Reference solutions sit in collapsed cells: Open them only after a genuine attempt.

The animations of the lecture run on the [lecture page](https://utkukose.github.io/ai-applications-in-engineering-NB-lecture/weeks/week-13/lecture.html), and the interactive lab with the self-assessment is available at <https://utkukose.github.io/ai-applications-in-engineering-NB-lecture/weeks/week-13/lab.html>. Cells that download public data need an internet connection. When a source is unreachable, the notebook falls back to a documented synthetic stand-in so that every later cell still runs.

## Overview

Engineering knowledge lives in text: standards, specifications, maintenance logs, test reports and code. Large language models process such text with the transformer architecture, whose self-attention lets every token weigh every other token [1]. Pretrained on vast corpora and tuned to follow instructions, they draft, summarise, translate and write code [2, 3, 4], but they also produce fluent statements that are false [5]. This week builds tokenisation, attention and a tiny transformer from scratch, connects a language task to retrieval so that answers rest on cited documents [6], surveys generative models for images and designs [7, 8], and sets out rules for responsible use in engineering work.

**Estimated study time:** 10 to 12 hours.

> **Final project.** The final project is due at the end of Week 14. The [project page](https://github.com/utkukose/ai-applications-in-engineering-NB-lecture/blob/main/exams/final/README.md) lists the deliverables and the evaluation criteria.

## Learning outcomes

By the end of the week, students are expected to explain tokenisation, embeddings and scaled dot-product attention, to compute attention weights for a small example, to describe how decoder-only language models are pretrained and tuned, to explain the effect of sampling temperature, to build and evaluate a retrieval step that grounds answers in documents, to name the main families of generative models, and to state the verification and confidentiality duties that apply when engineers use these tools.

## Python in this week

The lecture ends with Python step 13: Text, vectors and attention. It covers strings and their methods, counting words with Counter, regular expressions for quantities with units, JSON files, bag-of-words vectors with cosine similarity for retrieval, and softmax attention with a causal mask, applied to maintenance logs, specifications and word embeddings [1, 9]. The hands-on part of the notebook opens with the same step as runnable cells and short exercises with immediate feedback, and its later sections mark further constructs as Python moves.

## Text as engineering data

A large share of engineering knowledge is written: standards and codes, design reports, maintenance and incident logs, requirements, patents and source code. Natural language processing turns such text into data. Classic tasks include search and retrieval, classification of reports, extraction of quantities and entities, summarisation and translation. Information retrieval developed many of the tools, such as term weighting and vector space models, long before neural networks [10]. Large language models add the ability to generate text and code, which changes how engineers draft, search and program, and raises new questions about accuracy and responsibility.

## From words to vectors

A model first splits text into tokens. Whole words give huge vocabularies and fail on new words; single characters make sequences long. Byte-pair encoding starts from characters and repeatedly merges the most frequent adjacent pair into a new symbol, so that common words become single tokens while rare words split into meaningful pieces [11]. Each token is then mapped to a vector, its embedding. Word2vec showed that embeddings learned from co-occurrence capture semantic relations, so that words used in similar contexts receive similar vectors [12]. Sentence embeddings extend the idea to whole passages and support semantic search [13].

## Attention and the transformer

Attention was introduced to let a translation model look at the relevant words of the source sentence while producing each output word [14]. The transformer made attention the central operation [1]. Every token produces a query, a key and a value vector through learned linear maps. The attention weights of a token are the softmax of the dot products of its query with all keys, divided by the square root of the key dimension; its output is the weighted sum of the values. Several attention heads run in parallel and learn different relations, positional encodings inject word order, and residual connections and layer normalisation, as in the ResNets of Week 11, keep deep stacks trainable.

Two families dominate. Encoder models such as BERT see the whole text at once and are pretrained to fill in masked words, which suits classification and extraction [15]. Decoder models such as GPT are pretrained to predict the next token from the previous ones, with a causal mask that hides the future, and generate text one token at a time [2, 16]. Transformers now process images, sequences of measurements and molecules as well [17, 18].


> **Animation: Where does each token look?.** Select a token of the sentence to see its attention weights over the others. The embeddings here are small hand-made vectors, chosen for illustration rather than learned; the temperature scales the dot products before the softmax. Run it in the [web version of the lecture](https://utkukose.github.io/ai-applications-in-engineering-NB-lecture/weeks/week-13/lecture.html#anim-attention).


![Attention weights of one head of the tiny character-level transformer trained in the notebook. The causal mask leaves the upper triangle empty: No position may look at later characters.](https://utkukose.github.io/ai-applications-in-engineering-NB-lecture/weeks/week-13/figures/w13_fig1.png)

*Figure 13.1. Attention weights of one head of the tiny character-level transformer trained in the notebook. The causal mask leaves the upper triangle empty: No position may look at later characters.*




<details>
<summary><b>Check your understanding.</b> In scaled dot-product attention, why are the dot products divided by the square root of the key dimension?</summary>

A. To make the weights sum to one  
B. To keep the dot products from growing with the dimension, which would push the softmax into saturation  
C. To hide future tokens  
D. To reduce the number of parameters

**Answer: B.** Large dot products make the softmax nearly one-hot and its gradients tiny; the scaling keeps them in a useful range [1].

</details>

## Large language models

A language model assigns probabilities to the next token. Trained on hundreds of billions of tokens, decoder transformers develop broad abilities, and performance improves predictably with model size, data and compute, as empirical scaling laws describe [19]. GPT-3 showed that a large model can perform new tasks from a few examples in the prompt, without changing its weights [2]. Instruction tuning and reinforcement learning from human feedback then align models with what users ask for [3]. Prompts that ask for intermediate reasoning steps improve performance on multi-step problems [20], and models trained on code write and explain programs [4]. Such broadly trained models, adapted to many tasks, are called foundation models [21].

Generation samples tokens from the predicted distribution. A temperature below one sharpens the distribution towards the most likely tokens; a temperature above one flattens it and increases variety and the risk of nonsense. The model has no built-in notion of truth: It produces what is probable given its training, and fluent but unsupported statements, called hallucinations, are a known failure mode [5]. For an engineer, a model's answer about a load factor or a material property is a hypothesis to check, never a source.


<details>
<summary><b>Check your understanding.</b> A model&#x27;s next-token logits are 2.0, 1.0 and 0.0. What happens to the probability of the first token when the temperature is lowered from 1.0 to 0.5?</summary>

A. It decreases  
B. It increases, because dividing the logits by a smaller temperature enlarges their differences  
C. It stays the same  
D. It becomes exactly 0.5

**Answer: B.** Dividing by 0.5 doubles the gaps between logits, so the softmax concentrates more mass on the largest one.

</details>

## Retrieval-augmented generation

Retrieval-augmented generation combines a retriever with a generator [6]. The retriever searches a document collection, for example a company's standards or a project's reports, for passages relevant to the question. The generator then answers using those passages and can cite them. Grounding reduces hallucination, keeps answers current without retraining, and lets a reader verify each claim against its source. Retrieval can use sparse term weighting such as TF-IDF [10] or dense sentence embeddings [13]. Its quality can be measured like any classifier: For a set of questions with known relevant passages, the share of questions whose relevant passage appears among the top results is the hit rate.


![Cosine similarities between test questions and the passages of the notebook's small engineering knowledge base, computed with TF-IDF vectors. Bright cells on the diagonal mean the right passage is found first.](https://utkukose.github.io/ai-applications-in-engineering-NB-lecture/weeks/week-13/figures/w13_fig2.png)

*Figure 13.2. Cosine similarities between test questions and the passages of the notebook's small engineering knowledge base, computed with TF-IDF vectors. Bright cells on the diagonal mean the right passage is found first.*



<details>
<summary><b>Check your understanding.</b> What is the main engineering benefit of retrieval-augmented generation over asking a model directly?</summary>

A. It removes the need for any verification  
B. Answers are grounded in cited documents that can be checked and updated  
C. It makes the model smaller  
D. It guarantees correct answers

**Answer: B.** Retrieval ties answers to sources; checking them remains necessary, but it becomes possible [6].

</details>

## Generative models beyond text

Variational autoencoders learn a latent space from which new samples can be decoded [22]. Generative adversarial networks train a generator against a discriminator that tries to tell real from generated data [23]. Diffusion models learn to reverse a gradual noising process and now produce high-quality images; latent diffusion runs the process in a compressed space to make it efficient [7, 8]. In engineering, generative models propose designs, create synthetic training images for rare defects, and suggest candidate materials; deep learning has proposed hundreds of thousands of stable crystal structures [24], and language models have planned and run chemistry experiments with robotic equipment [25]. Every generated design still has to satisfy the physics, the codes and the tests.

## Responsible use in engineering work

Engineers remain accountable for their work, whatever tools they use. Four rules follow. Verify every factual or numerical claim against a primary source, a calculation or a test. Do not paste confidential designs, personal data or unpublished results into external services unless the organisation's policy allows it. Cite the sources that support the work, not the tool. And treat model inputs as an attack surface: Instructions hidden in documents can manipulate a model that reads them, and small perturbations can change the decisions of learned models [26]. The energy cost of training and running large models is also part of the engineering trade-off [27]. The course policy on the use of these tools is stated in the syllabus.

# Hands-on lab

The notebook learns byte-pair merges from a short engineering corpus, computes scaled dot-product attention with a causal mask in NumPy, and trains a tiny character-level transformer in PyTorch that generates text at different temperatures [1]. It then builds a TF-IDF retriever over a small knowledge base of engineering passages, measures its hit rate on test questions, and assembles grounded answers with citations [6, 10]. An optional cell connects an open instruction-tuned model when the libraries and a network connection are available.

## 0. Setup

In [ ]:
import math
import re
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
torch.manual_seed(0)

def check(name, value, expected, tol=1e-6):
    """Compare an exercise answer with the reference value and print feedback."""
    if value is None:
        print(f"[{name}] Not attempted yet. Replace None with your answer and run the cell again.")
        return
    try:
        ok = abs(float(value) - float(expected)) <= tol * max(1.0, abs(float(expected)))
    except (TypeError, ValueError):
        ok = value == expected
    print(f"[{name}] {'Correct.' if ok else 'Not yet. Check the logic and run again.'}")

## Python step 13: Text, vectors and attention

This part follows the Python step at the end of the lecture. Run the cells in order, compare each result with the explanation, and change values to see what happens. The exercises at the end of the step give immediate feedback.

### Strings

Text is a sequence of characters. Strings support indexing and slicing like lists, and their methods return transformed copies: `strip` removes surrounding spaces, `lower` converts to lower case, `split` breaks a string into a list of words, and `join` glues a list of strings together. `in` tests whether a piece of text occurs, and `count` counts its occurrences. Strings cannot be changed in place, so every method returns a new string.

In [ ]:
line = "  2026-03-14 PUMP-3: High vibration at bearing DE; replaced seal.  "
clean = line.strip()
print(clean[:10], "|", clean[11:17])
print(clean.lower().split())
print("vibration" in clean.lower(), clean.count("e"))
print(" / ".join(["seal", "bearing", "impeller"]))
print(len("İSTANBUL"), len("İSTANBUL".lower()))

The last line shows a detail that matters for Turkish text. Converting the capital dotted İ to lower case with the default Unicode rules produces an i followed by a combining dot, so the lower-case string is one character longer than the original. Language-aware processing, for example search in Turkish documents, has to treat such letters explicitly.

### Counting words

`collections.Counter` counts the elements of any sequence and reports the most common ones. The nested comprehension below turns several log entries into one list of words: The first `for` runs over the entries and the second over the words of each entry. `re.findall(r"[a-z]+", ...)` keeps runs of letters and drops digits and punctuation. A dictionary comprehension builds a dictionary in the same compact way, and `sorted` with a `key` orders items by a computed value.

In [ ]:
import re
from collections import Counter

logs = ["High vibration at pump 3, bearing replaced.",
        "Pump 3 seal leaking; seal replaced.",
        "Low flow at pump 1, strainer cleaned.",
        "Vibration normal after bearing replacement at pump 3."]
words = [w for entry in logs for w in re.findall(r"[a-z]+", entry.lower())]
counts = Counter(words)
print(counts.most_common(4))
repeated = {w: n for w, n in counts.items() if n > 1}
print(sorted(repeated, key=repeated.get, reverse=True))

Word counts of this kind are the starting point of the TF-IDF retrieval of this week, which weights each word by how rare it is across all documents.

### Patterns with regular expressions

Regular expressions describe patterns of text. `\d+` matches one or more digits, `(?:\.\d+)?` an optional decimal part, `\s*` optional spaces and `(MPa|kN|mm)` one of three units. Parentheses without `?:` capture the parts that `re.findall` returns, here pairs of value and unit taken from a specification sentence.

In [ ]:
spec = "Use C30/37 concrete with 28-day strength 38 MPa, cover 40 mm and a column load of 1250.5 kN."
pairs = re.findall(r"(\d+(?:\.\d+)?)\s*(MPa|kN|mm)", spec)
print(pairs)
print({unit: float(value) for value, unit in pairs})

### Saving and loading structured text

Files are opened with `open` inside a `with` block, which closes the file automatically. JSON stores lists and dictionaries as text that other programs and languages can read, a convenient format for a small knowledge base of passages such as the one used for retrieval in this week.

In [ ]:
import json

passages = [{"id": "P1", "text": "Strength falls as the water-cement ratio rises."},
            {"id": "P2", "text": "Clays plot above the A-line of the plasticity chart."}]
with open("kb_demo.json", "w", encoding="utf-8") as f:
    json.dump(passages, f, indent=1)
with open("kb_demo.json", encoding="utf-8") as f:
    loaded = json.load(f)
print(len(loaded), loaded[1]["id"], loaded[1]["text"])

### From counts to vectors: retrieval

Retrieval compares a question with stored passages. A bag-of-words vector counts how often each word of a fixed vocabulary occurs in a text. The cosine similarity of two vectors, their dot product divided by the product of their lengths, is 1 for texts with the same word proportions and 0 for texts without a common word. Stacking the vectors of all passages into a matrix turns the comparison with a question into one matrix-vector product. This is the retrieval step of retrieval-augmented generation in its simplest form; TF-IDF weighting refines it by giving rare words more influence.

In [ ]:
import numpy as np

vocab = sorted(set(words))
col = {w: i for i, w in enumerate(vocab)}


def bow(text):
    """Bag-of-words count vector of a text over the vocabulary."""
    v = np.zeros(len(vocab))
    for w in re.findall(r"[a-z]+", text.lower()):
        if w in col:
            v[col[w]] += 1
    return v


D = np.array([bow(entry) for entry in logs])          # one row per log entry
q = bow("bearing vibration")
sim = D @ q / (np.linalg.norm(D, axis=1) * np.linalg.norm(q))
print(D.shape, sim.round(2))
print("best match:", logs[int(np.argmax(sim))])

### Softmax and attention

Attention lets every token of a sequence gather information from the other tokens [1]. Each token has a query, a key and a value vector. The scores are the dot products of queries and keys, divided by the square root of their length. A softmax along each row turns the scores into weights that are positive and sum to one, and each output is the weighted mean of the values. In a transformer, learned matrices produce queries, keys and values from the token embeddings; the toy example below uses three hand-made embeddings directly, with "pump" and "impeller" deliberately similar.

In [ ]:
def softmax(s, axis=-1):
    """Softmax along an axis; subtracting the maximum avoids overflow."""
    e = np.exp(s - s.max(axis=axis, keepdims=True))
    return e / e.sum(axis=axis, keepdims=True)


emb = np.array([[1.0, 0.0, 1.0, 0.0],      # "pump"
                [0.0, 1.0, 0.0, 1.0],      # "seal"
                [1.0, 0.0, 0.9, 0.1]])     # "impeller"
Q, K, V = emb, emb, emb
scores = Q @ K.T / np.sqrt(emb.shape[1])
weights = softmax(scores, axis=1)
print(weights.round(2))
print((weights @ V).round(2))

A model that generates text one token at a time must not look at future tokens. A causal mask sets the scores above the diagonal to minus infinity before the softmax, so that every token attends only to itself and to earlier tokens, as in the decoders of generative language models [2].

In [ ]:
future = np.triu(np.ones((3, 3), dtype=bool), k=1)     # True above the diagonal
print(softmax(np.where(future, -np.inf, scores), axis=1).round(2))

**Quick check.** What does "pump seal leak".split() return?

A. ["pump", "seal", "leak"]  
B. "pumpsealleak"  
C. ["p", "u", "m", "p"]  
D. 3

<details><summary>Show the answer</summary>

**A.** Without an argument, split breaks the string at runs of white space and returns the words as a list.

</details>

### Exercises for Python step 13

**Exercise 13.1.** Count the different words in `logs`, after conversion to lower case and with digits and punctuation removed as above, and store the number in `py_unique_words`.

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _py13_words():
    import re
    lg = ["High vibration at pump 3, bearing replaced.", "Pump 3 seal leaking; seal replaced.",
          "Low flow at pump 1, strainer cleaned.", "Vibration normal after bearing replacement at pump 3."]
    return [w for e in lg for w in re.findall(r"[a-z]+", e.lower())]


def _py13_unique_words_reference():
    return len(set(_py13_words()))

In [ ]:
py_unique_words = None
check("Exercise 13.1", py_unique_words, _py13_unique_words_reference())

**Exercise 13.2.** Extract all values given in millimetres from the text "bars of 12 mm and 16 mm, cover 40 mm, span 5.2 m" with a regular expression, and store their sum in `py_mm_sum`.

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _py13_mm_sum_reference():
    import re
    return sum(float(v) for v in re.findall(r"(\d+(?:\.\d+)?)\s*mm", "bars of 12 mm and 16 mm, cover 40 mm, span 5.2 m"))

In [ ]:
py_mm_sum = None
check("Exercise 13.2", py_mm_sum, _py13_mm_sum_reference())

**Exercise 13.3.** Store the most common word of `logs` in `py_top_word`, taken from the result of `Counter`.

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _py13_top_word_reference():
    from collections import Counter
    return Counter(_py13_words()).most_common(1)[0][0]

In [ ]:
py_top_word = None
check("Exercise 13.3", py_top_word, _py13_top_word_reference())

**Exercise 13.4.** Compute the cosine similarity of the count vectors [1, 2, 0, 1] and [0, 1, 1, 1] and store it in `py_cosine`.

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _py13_cosine_reference():
    import numpy as np
    a, b = np.array([1.0, 2.0, 0.0, 1.0]), np.array([0.0, 1.0, 1.0, 1.0])
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

In [ ]:
py_cosine = None
check("Exercise 13.4", py_cosine, _py13_cosine_reference())

**Exercise 13.5.** Store the attention weight of "pump" (row 0) on "impeller" (column 2) from `weights` in `py_att_02`. Why is it larger than the weight on "seal"?

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _py13_att_02_reference():
    import numpy as np
    e = np.array([[1.0, 0.0, 1.0, 0.0], [0.0, 1.0, 0.0, 1.0], [1.0, 0.0, 0.9, 0.1]])
    s = e @ e.T / 2.0
    w = np.exp(s - s.max(axis=1, keepdims=True))
    return float((w / w.sum(axis=1, keepdims=True))[0, 2])

In [ ]:
py_att_02 = None
check("Exercise 13.5", py_att_02, _py13_att_02_reference())

## 1. Byte-pair encoding from scratch

**Python move: Counter and string replacement.** Each word is written as characters separated by spaces with an end marker `_`. The most frequent adjacent pair is merged, and the procedure repeats [11].

In [ ]:
CORPUS = 'The engineer inspects the pump every week. The bearing temperature rises when the lubricant is old. The inspector measures the crack width on the bridge deck. The concrete cube is tested after twenty eight days. The controller opens the valve when the level is low and closes the valve when the level is high. The sensor reports the vibration of the motor every second. The model predicts the strength of the concrete from the mix. The operator records the flow of the river after the rain. The robot follows the path around the machines on the factory floor. The drill visits every hole on the circuit board once. The camera checks the fabric for defects while the loom runs. The team compares the forecast with the measured demand every morning. The engineer checks the result before the report is sent. '
words = Counter(" ".join(w) + " _" for w in re.findall(r"[a-z]+", CORPUS.lower()))


def learn_bpe(words, n_merges):
    vocab, merges = dict(words), []
    for _ in range(n_merges):
        pairs = Counter()
        for w, c in vocab.items():
            s = w.split()
            for a, b in zip(s, s[1:]):
                pairs[a, b] += c
        if not pairs:
            break
        (a, b), count = pairs.most_common(1)[0]
        if count < 2:
            break
        merges.append((a, b))
        pattern = re.compile(r"(?<!\S)" + re.escape(a) + " " + re.escape(b) + r"(?!\S)")
        vocab = {pattern.sub(a + b, w): c for w, c in vocab.items()}
    return merges


def encode(word, merges):
    s = list(word) + ["_"]
    for a, b in merges:
        i, out = 0, []
        while i < len(s):
            if i < len(s) - 1 and s[i] == a and s[i + 1] == b:
                out.append(a + b); i += 2
            else:
                out.append(s[i]); i += 1
        s = out
    return s


merges = learn_bpe(words, 80)
print("first merges:", merges[:10])
for w in ["concrete", "bearing", "inspector", "hydrogen"]:
    print(f"{w:10s} -> {encode(w, merges)}")

## 2. Attention in NumPy

### Your turn, exercise 1

For the three tokens below, compute scaled dot-product attention softmax(Q K^T / sqrt(d)) V without a mask and store the sum of all elements of the output in `attention_sum` [1].

In [ ]:
Q = np.array([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])
K = np.array([[1.0, 0.5], [0.0, 1.0], [0.5, 0.5]])
V = np.array([[1.0, 2.0], [3.0, 0.0], [0.0, 1.0]])


def softmax(z, axis=-1):
    z = z - z.max(axis=axis, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=axis, keepdims=True)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _attention_sum_reference():
    return float((softmax(Q @ K.T / np.sqrt(K.shape[1])) @ V).sum())

In [ ]:
attention_sum = None   # your computation
check("Exercise 1", attention_sum, _attention_sum_reference(), tol=1e-9)

### Your turn, exercise 2

Logits of 2.0, 1.0 and 0.0 are turned into probabilities with a temperature T by softmax(logits / T). Store the probability of the first token at T = 0.5 in `p_first`.

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _p_first_reference():
    return float(softmax(np.array([2.0, 1.0, 0.0]) / 0.5)[0])

In [ ]:
p_first = None   # your computation
check("Exercise 2", p_first, _p_first_reference(), tol=1e-9)

## 3. A tiny character-level transformer

The model embeds characters and positions, applies two transformer layers with four attention heads and a causal mask, and predicts the next character. Trained for a few hundred steps on the short corpus, it learns spelling and frequent phrases, and nothing about engineering.

**Python move: a causal mask in PyTorch.** `nn.Transformer.generate_square_subsequent_mask(n)` returns a matrix with minus infinity above the diagonal, which blocks attention to later positions.

In [ ]:
chars = sorted(set(CORPUS))
stoi, itos = {c: i for i, c in enumerate(chars)}, {i: c for i, c in enumerate(chars)}
data = torch.tensor([stoi[c] for c in CORPUS])
CTX, D = 48, 64


class TinyGPT(nn.Module):
    def __init__(self, vocab, d=D, heads=4, layers=2, ctx=CTX):
        super().__init__()
        self.tok, self.pos = nn.Embedding(vocab, d), nn.Embedding(ctx, d)
        layer = nn.TransformerEncoderLayer(d, heads, dim_feedforward=4 * d, dropout=0.1, batch_first=True, norm_first=True)
        self.blocks = nn.TransformerEncoder(layer, layers)
        self.head = nn.Linear(d, vocab)

    def forward(self, idx):
        n = idx.shape[1]
        x = self.tok(idx) + self.pos(torch.arange(n))
        mask = nn.Transformer.generate_square_subsequent_mask(n)
        return self.head(self.blocks(x, mask=mask, is_causal=True))


model = TinyGPT(len(chars))
opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
losses = []
for step in range(400):
    ix = torch.randint(0, len(data) - CTX - 1, (32,))
    xb = torch.stack([data[i:i + CTX] for i in ix]); yb = torch.stack([data[i + 1:i + CTX + 1] for i in ix])
    loss = nn.functional.cross_entropy(model(xb).reshape(-1, len(chars)), yb.reshape(-1))
    opt.zero_grad(); loss.backward(); opt.step(); losses.append(loss.item())
print("parameters:", sum(p.numel() for p in model.parameters()), "final training loss:", round(float(np.mean(losses[-20:])), 3))

In [ ]:
@torch.no_grad()
def generate(prompt, n=120, temperature=1.0, seed=0):
    torch.manual_seed(seed); model.eval()
    idx = torch.tensor([[stoi.get(c, 0) for c in prompt]])
    for _ in range(n):
        logits = model(idx[:, -CTX:])[0, -1] / temperature
        nxt = torch.multinomial(torch.softmax(logits, -1), 1)
        idx = torch.cat([idx, nxt[None]], dim=1)
    return "".join(itos[int(i)] for i in idx[0])


for T in [0.3, 1.0, 1.8]:
    print(f"T = {T}: {generate('The engineer ', temperature=T)!r}\n")

## 4. Retrieval over an engineering knowledge base

Twelve short passages written for this course summarise ideas from earlier weeks. TF-IDF weights words that are frequent in a passage but rare in the collection [10]; cosine similarity ranks the passages for a question.

In [ ]:
KB = pd.DataFrame([('P1', 'Concrete strength', 'The compressive strength of concrete falls as the water-cement ratio rises. Strength grows with age at a decreasing rate, and standard tests report it at 28 days in MPa.'), ('P2', 'Soil classification', 'In the Unified Soil Classification System, a soil is fine grained when at least half of it passes the No. 200 sieve. Clays plot above the A-line of the plasticity chart and silts below it.'), ('P3', 'Bearing faults', 'An outer-race defect of a rolling bearing produces impacts at the ball pass frequency of the outer race. Envelope analysis reveals this frequency when the raw spectrum hides it.'), ('P4', 'Integral windup', 'When an actuator saturates, the integral term of a PID controller keeps growing. The stored integral causes overshoot after the saturation ends; anti-windup schemes limit the integral.'), ('P5', 'Fuzzy control', 'A Mamdani controller fuzzifies its inputs, evaluates rules with the minimum as AND, clips the output sets, aggregates them with the maximum and defuzzifies with the centroid.'), ('P6', 'Data leakage', 'Leakage occurs when information that is unavailable at prediction time enters training. Scaling fitted on all rows before splitting and features computed from the target are common causes.'), ('P7', 'Imbalanced classes', 'When failures are rare, accuracy is misleading. Precision-recall curves, class weights and thresholds chosen from the costs of misses and false alarms give a fairer picture.'), ('P8', 'Path planning', 'A* expands the node with the smallest sum of path cost and heuristic. With an admissible heuristic that never overestimates, the returned path is optimal.'), ('P9', 'Particle swarms', 'In particle swarm optimisation, each particle is pulled towards its own best position and the best position of the swarm. The inertia weight balances exploration and exploitation.'), ('P10', 'Transfer learning', 'A network pretrained on a large image dataset can be adapted to a small engineering dataset by retraining its last layers. Augmentation adds flipped and rotated copies of the images.'), ('P11', 'Forecast evaluation', 'Forecasts must be tested on data after the training period. The seasonal naive forecast, which repeats the value from one season earlier, is a mandatory baseline.'), ('P12', 'Anomaly detection', 'An isolation forest isolates unusual points with few random splits. Detectors trained at one operating condition raise false alarms when the load or speed changes.')], columns=["id", "title", "text"])
vectorizer = TfidfVectorizer(stop_words="english", sublinear_tf=True)
P = vectorizer.fit_transform(KB["title"] + ". " + KB["text"])


def retrieve(question, k=3):
    sims = cosine_similarity(vectorizer.transform([question]), P).ravel()
    order = np.argsort(sims)[::-1][:k]
    return [(KB.loc[i, "id"], KB.loc[i, "title"], round(float(sims[i]), 3)) for i in order]


def grounded_answer(question, threshold=0.1):
    hits = retrieve(question, k=1)
    pid, title, sim = hits[0]
    if sim < threshold:
        return "No passage supports an answer to this question."
    text = KB.set_index("id").loc[pid, "text"]
    return f"{text} [{pid}: {title}]"


q = "Why does a PID controller overshoot after the actuator saturates?"
print(retrieve(q))
print(grounded_answer(q))
print(grounded_answer("What is the boiling point of mercury?"))

### Your turn, exercise 3

Each question below comes with the id of the passage that answers it. Compute the hit rate at 1, the share of questions whose correct passage is ranked first, and store it in `hit_at_1`. Then write three questions of your own that use different words from the passages, for example synonyms or paraphrases, and test them. What would improve the misses: better wording, more passages or dense sentence embeddings [13]?

In [ ]:
TESTS = [("At what age is concrete strength usually reported?", "P1"), ("When is a soil called fine grained?", "P2"),
         ("Which analysis reveals the fault frequency of a bearing?", "P3"), ("What causes overshoot after actuator saturation?", "P4"),
         ("How does a Mamdani controller defuzzify?", "P5"), ("Why must scaling be fitted after splitting the data?", "P6"),
         ("Why is accuracy misleading for rare failures?", "P7"), ("When is the path returned by A* optimal?", "P8"),
         ("What balances exploration and exploitation in a swarm?", "P9"), ("How can a small image dataset use a pretrained network?", "P10"),
         ("Which baseline repeats last year's value?", "P11"), ("Why do anomaly detectors raise alarms after a load change?", "P12")]

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _hit_at_1_reference():
    return float(np.mean([retrieve(q, k=1)[0][0] == pid for q, pid in TESTS]))

In [ ]:
hit_at_1 = None   # your computation
check("Exercise 3", hit_at_1, _hit_at_1_reference(), tol=1e-9)

In [ ]:
sims = cosine_similarity(vectorizer.transform([q for q, _ in TESTS]), P)
plt.figure(figsize=(5.4, 4.6)); plt.imshow(sims, cmap="viridis")
plt.xticks(range(len(KB)), KB["id"], rotation=90, fontsize=7); plt.yticks(range(len(TESTS)), [pid for _, pid in TESTS], fontsize=7)
plt.xlabel("passage"); plt.ylabel("question (id of correct passage)"); plt.colorbar(label="cosine similarity"); plt.title("Retrieval similarities"); plt.show()

## 5. Optional: an open instruction-tuned model

If the `transformers` library and a network connection are available, the cell below loads a small open model and asks it to answer from the retrieved passage only. Leave `USE_MODEL = False` to skip it. Small models make mistakes; compare every sentence of the answer with the passage.

In [ ]:
USE_MODEL = False
if USE_MODEL:
    try:
        from transformers import pipeline
        generator = pipeline("text2text-generation", model="google/flan-t5-small")
        context = KB.set_index("id").loc[retrieve(q, k=1)[0][0], "text"]
        prompt = f"Answer the question using only the context.\nContext: {context}\nQuestion: {q}"
        print(generator(prompt, max_new_tokens=60)[0]["generated_text"])
    except Exception as error:
        print("Model unavailable:", type(error).__name__)

## Discipline challenges

Every department writes and reads text. Choose a row and adapt the notebook's retrieval section, or its tiny language model, to the documents named there.

| Department | Challenge |
|---|---|
| Computer Engineering | Retrieval over the documentation of a software library; evaluate code suggestions against unit tests [4]. |
| Civil Engineering | Retrieval over summaries of design code clauses written by the student; answer questions with clause citations. |
| Mechanical Engineering and Industrial Engineering | Classify maintenance log entries by failure mode with TF-IDF and a linear classifier. |
| Electrical and Electronics Engineering | Extract ratings and quantities from equipment datasheets and check them against the source. |
| Chemistry and Chemical Engineering | Summarise safety data sheet sections and verify every hazard statement against the original [25]. |
| Environmental Engineering | Search environmental impact report passages for mitigation measures by topic. |
| Food Engineering and Biology | Retrieval over open protocols for laboratory methods with a check of every quantity. |
| Geological Engineering, Geophysical Engineering, Mining and Earth Sciences Engineering | Index field notes or borehole log descriptions and query lithologies. |
| Physics, Mathematics and Statistics | Implement multi-head attention in NumPy and verify it against PyTorch's `nn.MultiheadAttention`. |
| Textile Engineering and Automotive Engineering | Cluster customer complaint texts by topic and relate the clusters to process or design causes. |

## Self-assessment and reflection

Take the self-assessment in the interactive lab: <https://utkukose.github.io/ai-applications-in-engineering-NB-lecture/weeks/week-13/lab.html> (tab: Check yourself). Then answer the reflection prompts:

1. Which documents of your field would you place in a retrieval system, and what would go wrong if an answer were taken without checking the passage?
2. The tiny transformer of the notebook produced text that looks like English. What does this show, and what does it not show, about understanding?
3. Write down your personal rules for using language models in coursework and engineering practice.

## Weekly task and submission

Build a small retrieval system for documents of your department that you are allowed to share, such as public standards summaries, course notes or open reports. Create at least ten passages and ten test questions with known answers, measure the hit rate at 1 and at 3 with TF-IDF, and discuss failure cases. If you also use a language model, show one grounded answer with citations and one answer that needed correction. Write about 500 words and cite at least three works from this week's references [1, 5, 6].

The weekly task is optional and supports self-learning and a personal portfolio. During an active semester, it can be sent together with the exported learning log to utkukose@sdu.edu.tr or utkukose@gmail.com. Optional work carries no separate weight, but the instructor may take it into account as a discretionary adjustment of the midterm component of the grade.

## Research and report assignment (optional)

**Language models in engineering practice.** Review how large language models are evaluated or used in one engineering activity, such as code generation, requirements engineering, standards compliance checking, maintenance log analysis or laboratory automation. Discuss accuracy, verification, confidentiality and accountability, with attention to documented failure modes [4, 5, 21, 25].

This research assignment is optional and supports self-learning. When the related weeks are followed within the course during an active semester, the report can be sent to utkukose@sdu.edu.tr or utkukose@gmail.com. It carries no separate weight, but the instructor may take it into account as a discretionary adjustment of the midterm component of the grade. Unless the assignment states otherwise, a report has 1500 to 2500 words, follows the structure of an academic paper, cites at least six scholarly or official sources in square brackets and uses the template in `exams/REPORT_TEMPLATE.md`.

## References

[1] Vaswani, A., Shazeer, N., Parmar, N., Uszkoreit, J., Jones, L., Gomez, A. N., Kaiser, Ł., & Polosukhin, I. (2017). Attention is all you need. In *Advances in Neural Information Processing Systems 30* (pp. 5998-6008). <https://arxiv.org/abs/1706.03762>

[2] Brown, T. B., Mann, B., Ryder, N., et al. (2020). Language models are few-shot learners. In *Advances in Neural Information Processing Systems 33* (pp. 1877-1901). <https://arxiv.org/abs/2005.14165>

[3] Ouyang, L., Wu, J., Jiang, X., et al. (2022). Training language models to follow instructions with human feedback. In *Advances in Neural Information Processing Systems 35* (pp. 27730-27744). <https://arxiv.org/abs/2203.02155>

[4] Chen, M., Tworek, J., Jun, H., et al. (2021). Evaluating large language models trained on code. arXiv preprint arXiv:2107.03374. <https://arxiv.org/abs/2107.03374>

[5] Ji, Z., Lee, N., Frieske, R., et al. (2023). Survey of hallucination in natural language generation. *ACM Computing Surveys*, *55*(12), 248. <https://doi.org/10.1145/3571730>

[6] Lewis, P., Perez, E., Piktus, A., et al. (2020). Retrieval-augmented generation for knowledge-intensive NLP tasks. In *Advances in Neural Information Processing Systems 33* (pp. 9459-9474). <https://arxiv.org/abs/2005.11401>

[7] Ho, J., Jain, A., & Abbeel, P. (2020). Denoising diffusion probabilistic models. In *Advances in Neural Information Processing Systems 33* (pp. 6840-6851). <https://arxiv.org/abs/2006.11239>

[8] Rombach, R., Blattmann, A., Lorenz, D., Esser, P., & Ommer, B. (2022). High-resolution image synthesis with latent diffusion models. In *2022 IEEE/CVF Conference on Computer Vision and Pattern Recognition (CVPR)* (pp. 10684-10695). <https://doi.org/10.1109/CVPR52688.2022.01042>

[9] Python Software Foundation (2026). *The Python Tutorial (Python 3 documentation)*. <https://docs.python.org/3/tutorial/>

[10] Manning, C. D., Raghavan, P., & Schütze, H. (2008). *Introduction to Information Retrieval*. Cambridge University Press.

[11] Sennrich, R., Haddow, B., & Birch, A. (2016). Neural machine translation of rare words with subword units. In *Proceedings of the 54th Annual Meeting of the Association for Computational Linguistics* (pp. 1715-1725). <https://doi.org/10.18653/v1/P16-1162>

[12] Mikolov, T., Chen, K., Corrado, G., & Dean, J. (2013). Efficient estimation of word representations in vector space. arXiv preprint arXiv:1301.3781. <https://arxiv.org/abs/1301.3781>

[13] Reimers, N., & Gurevych, I. (2019). Sentence-BERT: Sentence embeddings using Siamese BERT-networks. In *Proceedings of EMNLP-IJCNLP 2019* (pp. 3982-3992). <https://doi.org/10.18653/v1/D19-1410>

[14] Bahdanau, D., Cho, K., & Bengio, Y. (2015). Neural machine translation by jointly learning to align and translate. In *3rd International Conference on Learning Representations (ICLR 2015)*. <https://arxiv.org/abs/1409.0473>

[15] Devlin, J., Chang, M.-W., Lee, K., & Toutanova, K. (2019). BERT: Pre-training of deep bidirectional transformers for language understanding. In *Proceedings of NAACL-HLT 2019* (pp. 4171-4186). <https://doi.org/10.18653/v1/N19-1423>

[16] Radford, A., Wu, J., Child, R., Luan, D., Amodei, D., & Sutskever, I. (2019). *Language models are unsupervised multitask learners*. OpenAI.

[17] Dosovitskiy, A., Beyer, L., Kolesnikov, A., et al. (2021). An image is worth 16x16 words: Transformers for image recognition at scale. In *9th International Conference on Learning Representations (ICLR 2021)*. <https://arxiv.org/abs/2010.11929>

[18] Mousavi, S. M., Ellsworth, W. L., Zhu, W., Chuang, L. Y., & Beroza, G. C. (2020). Earthquake transformer: An attentive deep-learning model for simultaneous earthquake detection and phase picking. *Nature Communications*, *11*, 3952. <https://doi.org/10.1038/s41467-020-17591-w>

[19] Kaplan, J., McCandlish, S., Henighan, T., et al. (2020). Scaling laws for neural language models. arXiv preprint arXiv:2001.08361. <https://arxiv.org/abs/2001.08361>

[20] Wei, J., Wang, X., Schuurmans, D., et al. (2022). Chain-of-thought prompting elicits reasoning in large language models. In *Advances in Neural Information Processing Systems 35* (pp. 24824-24837). <https://arxiv.org/abs/2201.11903>

[21] Bommasani, R., Hudson, D. A., Adeli, E., et al. (2021). On the opportunities and risks of foundation models. arXiv preprint arXiv:2108.07258. <https://arxiv.org/abs/2108.07258>

[22] Kingma, D. P., & Welling, M. (2014). Auto-encoding variational Bayes. In *2nd International Conference on Learning Representations (ICLR 2014)*. <https://arxiv.org/abs/1312.6114>

[23] Goodfellow, I., Pouget-Abadie, J., Mirza, M., et al. (2014). Generative adversarial nets. In *Advances in Neural Information Processing Systems 27* (pp. 2672-2680). <https://arxiv.org/abs/1406.2661>

[24] Merchant, A., Batzner, S., Schoenholz, S. S., et al. (2023). Scaling deep learning for materials discovery. *Nature*, *624*(7990), 80-85. <https://doi.org/10.1038/s41586-023-06735-9>

[25] Boiko, D. A., MacKnight, R., Kline, B., & Gomes, G. (2023). Autonomous chemical research with large language models. *Nature*, *624*(7992), 570-578. <https://doi.org/10.1038/s41586-023-06792-0>

[26] Kose, U. (2019). Techniques for adversarial examples threatening the safety of artificial intelligence based systems. In *I. International Science and Innovation Congress (INSI Congress 2019)*. Pamukkale, Denizli, Türkiye. <https://arxiv.org/abs/1910.06907>

[27] Strubell, E., Ganesh, A., & McCallum, A. (2019). Energy and policy considerations for deep learning in NLP. In *Proceedings of the 57th Annual Meeting of the Association for Computational Linguistics* (pp. 3645-3650). <https://doi.org/10.18653/v1/P19-1355>